In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
## Loading train, val and test 5 min pred intervals
import os

DATA_DIR = "/content/drive/MyDrive/Threat-detection/gnn_edge_data"

TRAIN_FILE = os.path.join(
    DATA_DIR,
    "train_edges_5min_enriched.parquet"
)

VALIDATION_FILE = os.path.join(
    DATA_DIR,
    "validation_edges_5min_enriched.parquet"
)

TEST_FILE = os.path.join(
    DATA_DIR,
    "test_edges_5min_enriched.parquet"
)

In [ ]:
for file_path in [
    TRAIN_FILE,
    VALIDATION_FILE,
    TEST_FILE
]:
    print(
        os.path.basename(file_path),
        os.path.exists(file_path)
    )

train_edges_5min_enriched.parquet True
validation_edges_5min_enriched.parquet True
test_edges_5min_enriched.parquet True


In [ ]:
import pandas as pd

train_df = pd.read_parquet(TRAIN_FILE)
val_df = pd.read_parquet(VALIDATION_FILE)
test_df = pd.read_parquet(TEST_FILE)

In [ ]:
## sorting based on time

train_df = train_df.sort_values("window_start").reset_index(drop=True)
val_df = val_df.sort_values("window_start").reset_index(drop=True)
test_df = test_df.sort_values("window_start").reset_index(drop=True)

In [ ]:
## verifying data

datasets = {
    "Train": train_df,
    "Validation": val_df,
    "Test": test_df
}

for name, df in datasets.items():
    print(f"\n{name}")
    print(f"Shape: {df.shape}")
    print(
        "Time:",
        df["window_start"].min(),
        "to",
        df["window_start"].max()
    )
    print(
        "Labels:",
        df["binary_target"].value_counts().to_dict()
    )


Train
Shape: (150914, 18)
Time: 2021-12-17 07:00:00 to 2022-10-23 22:05:00
Labels: {0: 150135, 1: 779}

Validation
Shape: (295052, 18)
Time: 2024-02-28 04:00:00 to 2024-12-09 23:55:00
Labels: {0: 242139, 1: 52913}

Test
Shape: (232088, 18)
Time: 2025-05-23 00:00:00 to 2025-07-31 14:55:00
Labels: {0: 231130, 1: 958}


In [ ]:
train_df.head()

,data_split,window_start,src_ip_zeek,dest_ip_zeek,connection_count,total_duration,maximum_duration,total_orig_bytes,total_resp_bytes,total_orig_packets,total_resp_packets,tcp_count,udp_count,icmp_count,minimum_binary_target,binary_target,tactic_targets,mitre_targets
0,train,2021-12-17 07:00:00,143.88.1.52,224.0.0.252,2,1.880441e-01,9.402204e-02,160.0,0.0,4.0,0.0,0.0,2.0,0.0,0,0,None,None
1,train,2021-12-17 07:00:00,143.88.0.2,198.97.190.53,2,0.000000e+00,0.000000e+00,0.0,0.0,2.0,0.0,0.0,2.0,0.0,0,0,None,None
2,train,2021-12-17 07:00:00,143.88.0.2,192.36.148.17,2,0.000000e+00,0.000000e+00,0.0,0.0,2.0,0.0,0.0,2.0,0.0,0,0,None,None
3,train,2021-12-17 07:00:00,fe80::70d8:2461:35d9:c9ea,ff02::1:3,4,3.834381e-01,9.758186e-02,320.0,0.0,8.0,0.0,0.0,4.0,0.0,0,0,None,None
4,train,2021-12-17 07:00:00,143.88.0.2,143.88.255.10,2,3.246063e+06,1.623032e+06,363553120.0,363553120.0,6492020.0,6492020.0,0.0,0.0,2.0,0,0,None,None


In [ ]:
val_df.head()

,data_split,window_start,src_ip_zeek,dest_ip_zeek,connection_count,total_duration,maximum_duration,total_orig_bytes,total_resp_bytes,total_orig_packets,total_resp_packets,tcp_count,udp_count,icmp_count,minimum_binary_target,binary_target,tactic_targets,mitre_targets
0,validation,2024-02-28 04:00:00,143.88.4.11,143.88.5.17,2,0.003096,0.001567,0.0,0.0,6.0,0.0,2.0,0.0,0.0,1,1,[Reconnaissance],[T1595]
1,validation,2024-02-28 04:00:00,143.88.10.12,143.88.11.13,2,0.000000,0.000000,0.0,0.0,2.0,0.0,2.0,0.0,0.0,1,1,[Reconnaissance],[T1595]
2,validation,2024-02-28 04:00:00,143.88.10.12,143.88.11.8,2,0.000000,0.000000,0.0,0.0,2.0,0.0,2.0,0.0,0.0,1,1,[Reconnaissance],[T1595]
3,validation,2024-02-28 04:00:00,143.88.10.12,143.88.11.5,2,0.000000,0.000000,0.0,0.0,2.0,0.0,2.0,0.0,0.0,1,1,[Reconnaissance],[T1595]
4,validation,2024-02-28 04:00:00,143.88.4.11,143.88.5.10,1,0.003054,0.003054,0.0,0.0,4.0,4.0,1.0,0.0,0.0,1,1,[Reconnaissance],[T1595]


In [ ]:
test_df.head()

,data_split,window_start,src_ip_zeek,dest_ip_zeek,connection_count,total_duration,maximum_duration,total_orig_bytes,total_resp_bytes,total_orig_packets,total_resp_packets,tcp_count,udp_count,icmp_count,minimum_binary_target,binary_target,tactic_targets,mitre_targets
0,test,2025-05-23,143.88.4.13,143.88.255.10,2,0.000024,0.000015,182.0,0.0,4.0,0.0,0.0,2.0,0.0,0,0,None,None
1,test,2025-05-23,143.88.10.13,143.88.10.255,1,4.005710,4.005710,200.0,0.0,4.0,0.0,0.0,1.0,0.0,0,0,None,None
2,test,2025-05-23,143.88.5.10,143.88.255.10,1,0.000014,0.000014,76.0,0.0,2.0,0.0,0.0,1.0,0.0,0,0,None,None
3,test,2025-05-23,143.88.0.18,143.88.255.10,2,0.000061,0.000046,182.0,0.0,4.0,0.0,0.0,2.0,0.0,0,0,None,None
4,test,2025-05-23,143.88.3.13,143.88.255.10,2,0.000028,0.000014,182.0,0.0,4.0,0.0,0.0,2.0,0.0,0,0,None,None


In [ ]:
train_df.columns

Index(['data_split', 'window_start', 'src_ip_zeek', 'dest_ip_zeek',
       'connection_count', 'total_duration', 'maximum_duration',
       'total_orig_bytes', 'total_resp_bytes', 'total_orig_packets',
       'total_resp_packets', 'tcp_count', 'udp_count', 'icmp_count',
       'minimum_binary_target', 'binary_target', 'tactic_targets',
       'mitre_targets'],
      dtype='object')

In [ ]:
import torch

print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

GPU available: True
GPU: NVIDIA L4


### Scaling data

In [ ]:
## creating scaled features

import numpy as np
import pandas as pd

log_columns = [
    "connection_count",
    "total_duration",
    "maximum_duration",
    "total_orig_bytes",
    "total_resp_bytes",
    "total_orig_packets",
    "total_resp_packets"
]

def create_edge_features(df):

    features = pd.DataFrame(index=df.index)

    for column in log_columns:
        features[f"log_{column}"] = np.log1p(
            df[column].fillna(0).clip(lower=0)
        )

    denominator = df["connection_count"].clip(lower=1)

    features["tcp_ratio"] = (
        df["tcp_count"] / denominator
    )

    features["udp_ratio"] = (
        df["udp_count"] / denominator
    )

    features["icmp_ratio"] = (
        df["icmp_count"] / denominator
    )

    return features

bytes and connection counts can be extremely large so reducing their scale by log1p.

For tcp, udp, icmp protocols we are calculating proportions based on total connections.

In [ ]:
train_features = create_edge_features(train_df)
val_features = create_edge_features(val_df)
test_features = create_edge_features(test_df)

In [ ]:
train_features.head()

,log_connection_count,log_total_duration,log_maximum_duration,log_total_orig_bytes,log_total_resp_bytes,log_total_orig_packets,log_total_resp_packets,tcp_ratio,udp_ratio,icmp_ratio
0,1.098612,0.172308,0.089861,5.081404,0.000000,1.609438,0.000000,0.0,1.0,0.0
1,1.098612,0.000000,0.000000,0.000000,0.000000,1.098612,0.000000,0.0,1.0,0.0
2,1.098612,0.000000,0.000000,0.000000,0.000000,1.098612,0.000000,0.0,1.0,0.0
3,1.609438,0.324572,0.093109,5.771441,0.000000,2.197225,0.000000,0.0,1.0,0.0
4,1.098612,14.992954,14.299807,19.711436,19.711436,15.686084,15.686084,0.0,0.0,1.0


**Scaling using train data for numeric variables**

In [ ]:
from sklearn.preprocessing import StandardScaler

edge_scaler = StandardScaler()

X_train = edge_scaler.fit_transform(
    train_features
).astype("float32")

X_val = edge_scaler.transform(
    val_features
).astype("float32")

X_test = edge_scaler.transform(
    test_features
).astype("float32")

In [ ]:
print("Feature names:")
print(train_features.columns.tolist())

print("\nShapes:")
print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nInvalid values:")
print("Train:", (~np.isfinite(X_train)).sum())
print("Validation:", (~np.isfinite(X_val)).sum())
print("Test:", (~np.isfinite(X_test)).sum())

Feature names:
['log_connection_count', 'log_total_duration', 'log_maximum_duration', 'log_total_orig_bytes', 'log_total_resp_bytes', 'log_total_orig_packets', 'log_total_resp_packets', 'tcp_ratio', 'udp_ratio', 'icmp_ratio']

Shapes:
Train: (150914, 10)
Validation: (295052, 10)
Test: (232088, 10)

Invalid values:
Train: 0
Validation: 0
Test: 0


### Creating one example graph

In [ ]:
## selecting the first training interval containing an attack and collect its previous one-hour context.

attack_times = (
    train_df.loc[
        train_df["binary_target"] == 1,
        "window_start"
    ]
    .drop_duplicates()
    .sort_values()
)

In [ ]:
attack_times

,window_start
150135,2022-02-10 03:55:00
150136,2022-02-10 04:00:00
150137,2022-02-10 04:05:00
150138,2022-02-10 04:10:00
150139,2022-02-10 04:15:00
...,...
150909,2022-10-18 22:15:00
150910,2022-10-19 17:45:00
150911,2022-10-19 17:50:00
150912,2022-10-20 21:55:00


In [ ]:
target_time = None

for candidate_time in attack_times:

    context_start = candidate_time - pd.Timedelta(hours=1)

    context_mask = (
        (train_df["window_start"] >= context_start) &
        (train_df["window_start"] < candidate_time)
    )

    if context_mask.any():
        target_time = candidate_time
        break

In [ ]:
## creating context and target sections

context_start = target_time - pd.Timedelta(hours=1)

context_df = train_df[
    (train_df["window_start"] >= context_start) &
    (train_df["window_start"] < target_time)
].copy()

target_df = train_df[
    train_df["window_start"] == target_time
].copy()

In [ ]:
# examining the example

context_nodes = set(
    context_df["src_ip_zeek"]
).union(
    context_df["dest_ip_zeek"]
)

target_nodes = set(
    target_df["src_ip_zeek"]
).union(
    target_df["dest_ip_zeek"]
)

print("Context:")
print(context_start, "to", target_time)

print("\nPrediction interval:")
print(
    target_time,
    "to",
    target_time + pd.Timedelta(minutes=5)
)

print("\nContext edge records:", len(context_df))

print(
    "Unique context IP pairs:",
    context_df[
        ["src_ip_zeek", "dest_ip_zeek"]
    ].drop_duplicates().shape[0]
)

print("Context nodes:", len(context_nodes))
print("Target edges:", len(target_df))

print(
    "Target labels:",
    target_df["binary_target"].value_counts().to_dict()
)

print(
    "New nodes in target interval:",
    len(target_nodes - context_nodes)
)

Context:
2022-02-10 03:00:00 to 2022-02-10 04:00:00

Prediction interval:
2022-02-10 04:00:00 to 2022-02-10 04:05:00

Context edge records: 1
Unique context IP pairs: 1
Context nodes: 2
Target edges: 1
Target labels: {1: 1}
New nodes in target interval: 0


In [ ]:
## selecting attack intervals with at least 20 context edges

target_time = None

for candidate_time in attack_times:

    candidate_start = (
        candidate_time - pd.Timedelta(hours=1)
    )

    candidate_context = train_df[
        (train_df["window_start"] >= candidate_start) &
        (train_df["window_start"] < candidate_time)
    ]

    unique_pairs = candidate_context[
        ["src_ip_zeek", "dest_ip_zeek"]
    ].drop_duplicates().shape[0]

    if unique_pairs >= 20:
        target_time = candidate_time
        break

In [ ]:
context_start = target_time - pd.Timedelta(hours=1)

context_df = train_df[
    (train_df["window_start"] >= context_start) &
    (train_df["window_start"] < target_time)
].copy()

target_df = train_df[
    train_df["window_start"] == target_time
].copy()

In [ ]:
# examining the example

context_nodes = set(
    context_df["src_ip_zeek"]
).union(
    context_df["dest_ip_zeek"]
)

target_nodes = set(
    target_df["src_ip_zeek"]
).union(
    target_df["dest_ip_zeek"]
)

print("Context:")
print(context_start, "to", target_time)

print("\nPrediction interval:")
print(
    target_time,
    "to",
    target_time + pd.Timedelta(minutes=5)
)

print("\nContext edge records:", len(context_df))

print(
    "Unique context IP pairs:",
    context_df[
        ["src_ip_zeek", "dest_ip_zeek"]
    ].drop_duplicates().shape[0]
)

print("Context nodes:", len(context_nodes))
print("Target edges:", len(target_df))

print(
    "Target labels:",
    target_df["binary_target"].value_counts().to_dict()
)

print(
    "New nodes in target interval:",
    len(target_nodes - context_nodes)
)

Context:
2022-02-10 16:50:00 to 2022-02-10 17:50:00

Prediction interval:
2022-02-10 17:50:00 to 2022-02-10 17:55:00

Context edge records: 257
Unique context IP pairs: 257
Context nodes: 257
Target edges: 4
Target labels: {1: 4}
New nodes in target interval: 0


In [ ]:
%pip install -q torch-geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 72.3 MB/s eta 0:00:00


In [ ]:
import torch
from torch_geometric.data import Data

In [ ]:
## creating node mapping

all_ips = pd.unique(
    pd.concat([
        context_df["src_ip_zeek"],
        context_df["dest_ip_zeek"],
        target_df["src_ip_zeek"],
        target_df["dest_ip_zeek"]
    ])
)

node_to_id = {
    ip: index
    for index, ip in enumerate(all_ips)
}

num_nodes = len(node_to_id)

print("Number of nodes:", num_nodes)

Number of nodes: 257


In [ ]:
## node features

node_features = np.zeros(
    (num_nodes, 6),
    dtype=np.float32
)

for row in context_df.itertuples():

    src = node_to_id[row.src_ip_zeek]
    dst = node_to_id[row.dest_ip_zeek]

    count = row.connection_count

    node_features[src, 0] += count
    node_features[dst, 1] += count

    node_features[src, 2] += row.total_orig_bytes
    node_features[src, 3] += row.total_resp_bytes

    node_features[dst, 2] += row.total_resp_bytes
    node_features[dst, 3] += row.total_orig_bytes

    node_features[src, 4] += row.total_orig_packets
    node_features[src, 5] += row.total_resp_packets

    node_features[dst, 4] += row.total_resp_packets
    node_features[dst, 5] += row.total_orig_packets

In [ ]:
## reducing large values

node_features = np.log1p(node_features)

x = torch.tensor(
    node_features,
    dtype=torch.float32
)

In [ ]:
## creating conext connections

context_pairs = context_df[
    ["src_ip_zeek", "dest_ip_zeek"]
].drop_duplicates()

In [ ]:
context_sources = [
    node_to_id[ip]
    for ip in context_pairs["src_ip_zeek"]
]

context_destinations = [
    node_to_id[ip]
    for ip in context_pairs["dest_ip_zeek"]
]

In [ ]:
## adding connection in both directions so info can move between both computers

edge_index = torch.tensor(
    [
        context_sources + context_destinations,
        context_destinations + context_sources
    ],
    dtype=torch.long
)

In [ ]:
## creating target edges
target_edge_index = torch.tensor(
    [
        [
            node_to_id[ip]
            for ip in target_df["src_ip_zeek"]
        ],
        [
            node_to_id[ip]
            for ip in target_df["dest_ip_zeek"]
        ]
    ],
    dtype=torch.long
)

In [ ]:
## getting prepared features for these target edges

target_indices = target_df.index.to_numpy()

target_edge_attr = torch.tensor(
    X_train[target_indices],
    dtype=torch.float32
)

target_labels = torch.tensor(
    target_df["binary_target"].to_numpy(),
    dtype=torch.float32
)

In [ ]:
## creating graph object

example_graph = Data(
    x=x,
    edge_index=edge_index
)

example_graph.target_edge_index = target_edge_index
example_graph.target_edge_attr = target_edge_attr
example_graph.y = target_labels
example_graph.window_start = str(target_time)

In [ ]:
## inspecting the graph

print(example_graph)

print("Node features:", example_graph.x.shape)
print("Context connections:", example_graph.edge_index.shape)
print("Target connections:", example_graph.target_edge_index.shape)
print("Target edge features:", example_graph.target_edge_attr.shape)
print("Targets:", example_graph.y)

Data(x=[257, 6], edge_index=[2, 514], target_edge_index=[2, 4], target_edge_attr=[4, 10], y=[4], window_start='2022-02-10 17:50:00')
Node features: torch.Size([257, 6])
Context connections: torch.Size([2, 514])
Target connections: torch.Size([2, 4])
Target edge features: torch.Size([4, 10])
Targets: tensor([1., 1., 1., 1.])


In [ ]:
## creating all graphs

from bisect import bisect_left

def create_time_lookup(df):

    groups = {
        pd.Timestamp(time): indices
        for time, indices in
        df.groupby("window_start").indices.items()
    }

    times = sorted(groups.keys())

    return groups, times

In [ ]:
train_groups, train_times = create_time_lookup(train_df)
val_groups, val_times = create_time_lookup(val_df)
test_groups, test_times = create_time_lookup(test_df)

print("Train graphs:", len(train_times))
print("Validation graphs:", len(val_times))
print("Test graphs:", len(test_times))

Train graphs: 9300
Validation graphs: 10560
Test graphs: 5326


In [ ]:
## building graph
def build_graph(
    df,
    feature_matrix,
    groups,
    times,
    target_time
):
    # Current five-minute target rows
    target_indices = groups[target_time]
    target = df.iloc[target_indices]

    # Previous one-hour context
    context_start = target_time - pd.Timedelta(hours=1)

    left = bisect_left(times, context_start)
    right = bisect_left(times, target_time)

    context_times = times[left:right]

    if context_times:
        context_indices = np.concatenate([
            groups[time] for time in context_times
        ])
        context = df.iloc[context_indices]
    else:
        context = df.iloc[0:0]

    # Nodes appearing in context or target
    all_ips = pd.unique(
        pd.concat([
            context["src_ip_zeek"],
            context["dest_ip_zeek"],
            target["src_ip_zeek"],
            target["dest_ip_zeek"]
        ])
    )

    node_to_id = {
        ip: index
        for index, ip in enumerate(all_ips)
    }

    node_features = np.zeros(
        (len(all_ips), 6),
        dtype=np.float32
    )

    # Build node history features
    if len(context) > 0:

        src = context["src_ip_zeek"].map(
            node_to_id
        ).to_numpy()

        dst = context["dest_ip_zeek"].map(
            node_to_id
        ).to_numpy()

        count = context[
            "connection_count"
        ].to_numpy()

        orig_bytes = context[
            "total_orig_bytes"
        ].to_numpy()

        resp_bytes = context[
            "total_resp_bytes"
        ].to_numpy()

        orig_packets = context[
            "total_orig_packets"
        ].to_numpy()

        resp_packets = context[
            "total_resp_packets"
        ].to_numpy()

        np.add.at(node_features[:, 0], src, count)
        np.add.at(node_features[:, 1], dst, count)

        np.add.at(node_features[:, 2], src, orig_bytes)
        np.add.at(node_features[:, 3], src, resp_bytes)

        np.add.at(node_features[:, 2], dst, resp_bytes)
        np.add.at(node_features[:, 3], dst, orig_bytes)

        np.add.at(node_features[:, 4], src, orig_packets)
        np.add.at(node_features[:, 5], src, resp_packets)

        np.add.at(node_features[:, 4], dst, resp_packets)
        np.add.at(node_features[:, 5], dst, orig_packets)

    node_features = np.log1p(node_features)

    # Context connections
    context_pairs = context[
        ["src_ip_zeek", "dest_ip_zeek"]
    ].drop_duplicates()

    context_src = context_pairs[
        "src_ip_zeek"
    ].map(node_to_id).to_numpy()

    context_dst = context_pairs[
        "dest_ip_zeek"
    ].map(node_to_id).to_numpy()

    edge_index = torch.tensor(
        np.vstack([
            np.concatenate([context_src, context_dst]),
            np.concatenate([context_dst, context_src])
        ]),
        dtype=torch.long
    )

    # Target connections
    target_src = target[
        "src_ip_zeek"
    ].map(node_to_id).to_numpy()

    target_dst = target[
        "dest_ip_zeek"
    ].map(node_to_id).to_numpy()

    target_edge_index = torch.tensor(
        np.vstack([target_src, target_dst]),
        dtype=torch.long
    )

    graph = Data(
        x=torch.tensor(
            node_features,
            dtype=torch.float32
        ),
        edge_index=edge_index
    )

    graph.target_edge_index = target_edge_index

    graph.target_edge_attr = torch.tensor(
        feature_matrix[target_indices],
        dtype=torch.float32
    )

    graph.y = torch.tensor(
        target["binary_target"].to_numpy(),
        dtype=torch.float32
    )

    graph.window_start = str(target_time)

    return graph

In [ ]:
test_example = build_graph(
    train_df,
    X_train,
    train_groups,
    train_times,
    target_time
)

print(test_example)

Data(x=[257, 6], edge_index=[2, 514], target_edge_index=[2, 4], target_edge_attr=[4, 10], y=[4], window_start='2022-02-10 17:50:00')


### Generating training graph

In [ ]:
from tqdm.auto import tqdm

def build_all_graphs(
    df,
    feature_matrix,
    groups,
    times
):
    graphs = []

    for target_time in tqdm(times):

        graph = build_graph(
            df,
            feature_matrix,
            groups,
            times,
            target_time
        )

        graph.target_row_index = torch.tensor(
            groups[target_time],
            dtype=torch.long
        )

        graphs.append(graph)

    return graphs

In [ ]:
train_graphs = build_all_graphs(
    train_df,
    X_train,
    train_groups,
    train_times
)

  0%|          | 0/9300 [00:00<?, ?it/s]

In [ ]:
## verifying generated graphs

all_train_labels = torch.cat([
    graph.y for graph in train_graphs
])

print("Number of graphs:", len(train_graphs))

print(
    "Total target edges:",
    len(all_train_labels)
)

print(
    "Normal target edges:",
    int((all_train_labels == 0).sum())
)

print(
    "Attack target edges:",
    int((all_train_labels == 1).sum())
)

print(
    "Average nodes per graph:",
    sum(graph.num_nodes for graph in train_graphs)
    / len(train_graphs)
)

print(
    "Average context connections:",
    sum(graph.edge_index.shape[1] for graph in train_graphs)
    / len(train_graphs)
)

Number of graphs: 9300
Total target edges: 150914
Normal target edges: 150135
Attack target edges: 779
Average nodes per graph: 54.98924731182796
Average context connections: 110.6268817204301


In [ ]:
## generating validation graphs

val_graphs = build_all_graphs(
    val_df,
    X_val,
    val_groups,
    val_times
)

  0%|          | 0/10560 [00:00<?, ?it/s]

In [ ]:
## generating test graphs

test_graphs = build_all_graphs(
    test_df,
    X_test,
    test_groups,
    test_times
)

  0%|          | 0/5326 [00:00<?, ?it/s]

In [ ]:
## verifying splits

for name, graphs in {
    "Train": train_graphs,
    "Validation": val_graphs,
    "Test": test_graphs
}.items():

    labels = torch.cat([
        graph.y for graph in graphs
    ])

    print(f"\n{name}")
    print("Graphs:", len(graphs))
    print("Target edges:", len(labels))
    print("Normal:", int((labels == 0).sum()))
    print("Attack:", int((labels == 1).sum()))


Train
Graphs: 9300
Target edges: 150914
Normal: 150135
Attack: 779

Validation
Graphs: 10560
Target edges: 295052
Normal: 242139
Attack: 52913

Test
Graphs: 5326
Target edges: 232088
Normal: 231130
Attack: 958


In [ ]:
for graphs in [train_graphs, val_graphs, test_graphs]:
    for graph in graphs:
        graph.target_row_ids = graph.target_row_index
        del graph.target_row_index

In [ ]:
## Saving graph datasets

import joblib

GRAPH_DIR = os.path.join(
    DATA_DIR,
    "prepared_graphs"
)

os.makedirs(GRAPH_DIR, exist_ok=True)

In [ ]:
torch.save(
    train_graphs,
    os.path.join(GRAPH_DIR, "train_graphs.pt")
)

torch.save(
    val_graphs,
    os.path.join(GRAPH_DIR, "validation_graphs.pt")
)

torch.save(
    test_graphs,
    os.path.join(GRAPH_DIR, "test_graphs.pt")
)

In [ ]:
## saving scaler and feature names

joblib.dump(
    edge_scaler,
    os.path.join(GRAPH_DIR, "edge_scaler.joblib")
)

joblib.dump(
    train_features.columns.tolist(),
    os.path.join(GRAPH_DIR, "edge_feature_names.joblib")
)

['/content/drive/MyDrive/Threat-detection/gnn_edge_data/prepared_graphs/edge_feature_names.joblib']

### Baseline GNN

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import SAGEConv
from torch_geometric.loader import DataLoader

In [ ]:
## Creating data loaders

train_loader = DataLoader(
    train_graphs,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_graphs,
    batch_size=128,
    shuffle=False
)

test_loader = DataLoader(
    test_graphs,
    batch_size=128,
    shuffle=False
)

In [ ]:
## defining the model

class EdgeGraphSAGE(nn.Module):

    def __init__(
        self,
        node_features=6,
        edge_features=10,
        hidden_size=64
    ):
        super().__init__()

        self.conv1 = SAGEConv(
            node_features,
            hidden_size
        )

        self.conv2 = SAGEConv(
            hidden_size,
            hidden_size
        )

        self.edge_classifier = nn.Sequential(
            nn.Linear(
                hidden_size * 2 + edge_features,
                hidden_size
            ),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_size, 1)
        )

    def forward(self, data):

        node_embeddings = F.relu(
            self.conv1(
                data.x,
                data.edge_index
            )
        )

        node_embeddings = F.dropout(
            node_embeddings,
            p=0.2,
            training=self.training
        )

        node_embeddings = F.relu(
            self.conv2(
                node_embeddings,
                data.edge_index
            )
        )

        source, destination = (
            data.target_edge_index
        )

        edge_input = torch.cat(
            [
                node_embeddings[source],
                node_embeddings[destination],
                data.target_edge_attr
            ],
            dim=1
        )

        logits = self.edge_classifier(
            edge_input
        ).squeeze(-1)

        return logits

In [ ]:
## testing one forward pass

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = EdgeGraphSAGE().to(device)

sample_batch = next(iter(train_loader)).to(device)

with torch.no_grad():
    sample_logits = model(sample_batch)

print("Device:", device)
print("Prediction shape:", sample_logits.shape)
print("Target shape:", sample_batch.y.shape)

Device: cuda
Prediction shape: torch.Size([1064])
Target shape: torch.Size([1064])


In [ ]:
### Model Training
## giving more weights to attack samples as they are fewer

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score
)

torch.manual_seed(42)
np.random.seed(42)

normal_count = 150135
attack_count = 779

positive_weight = np.sqrt(normal_count / attack_count)

print("Attack weight:", positive_weight)

Attack weight: 13.882645865465657


In [ ]:
loss_function = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor(
        positive_weight,
        dtype=torch.float32,
        device=device
    )
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)

In [ ]:
def train_one_epoch():

    model.train()

    total_loss = 0
    total_targets = 0

    for batch in train_loader:

        batch = batch.to(device)

        optimizer.zero_grad()

        logits = model(batch)

        loss = loss_function(
            logits,
            batch.y.float()
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0
        )

        optimizer.step()

        total_loss += (
            loss.item() * batch.y.numel()
        )

        total_targets += batch.y.numel()

    return total_loss / total_targets

In [ ]:
## validation function

@torch.no_grad()
def evaluate_model(loader):

    model.eval()

    labels = []
    probabilities = []
    total_loss = 0
    total_targets = 0

    for batch in loader:

        batch = batch.to(device)

        logits = model(batch)

        loss = loss_function(
            logits,
            batch.y.float()
        )

        probs = torch.sigmoid(logits)

        labels.append(
            batch.y.cpu().numpy()
        )

        probabilities.append(
            probs.cpu().numpy()
        )

        total_loss += (
            loss.item() * batch.y.numel()
        )

        total_targets += batch.y.numel()

    labels = np.concatenate(labels)
    probabilities = np.concatenate(probabilities)

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    return {
        "loss": total_loss / total_targets,
        "pr_auc": average_precision_score(
            labels,
            probabilities
        ),
        "roc_auc": roc_auc_score(
            labels,
            probabilities
        ),
        "precision": precision_score(
            labels,
            predictions,
            zero_division=0
        ),
        "recall": recall_score(
            labels,
            predictions,
            zero_division=0
        ),
        "f1": f1_score(
            labels,
            predictions,
            zero_division=0
        )
    }

In [ ]:
## training with early stopping

## We select the model using validation PR-AUC. The 2025 test set remains untouched.

MODEL_DIR = os.path.join(
    DATA_DIR,
    "models"
)

os.makedirs(MODEL_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "binary_graphsage_best.pt"
)

In [ ]:
best_pr_auc = -1
epochs_without_improvement = 0
patience = 5

history = []

for epoch in range(1, 21):

    train_loss = train_one_epoch()
    val_metrics = evaluate_model(val_loader)

    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        **val_metrics
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_metrics['loss']:.4f} | "
        f"PR-AUC: {val_metrics['pr_auc']:.4f} | "
        f"F1: {val_metrics['f1']:.4f} | "
        f"Recall: {val_metrics['recall']:.4f}"
    )

    if val_metrics["pr_auc"] > best_pr_auc:

        best_pr_auc = val_metrics["pr_auc"]
        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            BEST_MODEL_PATH
        )

    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print("Early stopping")
        break

Epoch 01 | Train Loss: 0.1167 | Val Loss: 3.7467 | PR-AUC: 0.5661 | F1: 0.3974 | Recall: 0.7546
Epoch 02 | Train Loss: 0.0347 | Val Loss: 12.3065 | PR-AUC: 0.2909 | F1: 0.3217 | Recall: 0.6030
Epoch 03 | Train Loss: 0.0287 | Val Loss: 15.1959 | PR-AUC: 0.4583 | F1: 0.3454 | Recall: 0.4889
Epoch 04 | Train Loss: 0.0181 | Val Loss: 14.8173 | PR-AUC: 0.3831 | F1: 0.3196 | Recall: 0.5377
Epoch 05 | Train Loss: 0.0095 | Val Loss: 17.4746 | PR-AUC: 0.3574 | F1: 0.3117 | Recall: 0.5320
Epoch 06 | Train Loss: 0.0097 | Val Loss: 20.8094 | PR-AUC: 0.4150 | F1: 0.3292 | Recall: 0.4961
Early stopping


In [ ]:
from torch_geometric.loader import DataLoader

train_eval_loader = DataLoader(
    train_graphs,
    batch_size=128,
    shuffle=False
)

# Load the best saved model
saved = torch.load(BEST_MODEL_PATH, map_location=device)

if "model_state_dict" in saved:
    model.load_state_dict(saved["model_state_dict"])
else:
    model.load_state_dict(saved)

train_metrics = evaluate_model(train_eval_loader)
val_metrics = evaluate_model(val_loader)

print("Train F1:", round(train_metrics["f1"], 4))
print("Validation F1:", round(val_metrics["f1"], 4))

print("Train PR-AUC:", round(train_metrics["pr_auc"], 4))
print("Validation PR-AUC:", round(val_metrics["pr_auc"], 4))

Train F1: 0.674
Validation F1: 0.3974
Train PR-AUC: 0.7072
Validation PR-AUC: 0.5661


In [ ]:
import numpy as np
import torch
from sklearn.metrics import precision_recall_curve, average_precision_score

# Load the best epoch
model.load_state_dict(
    torch.load(BEST_MODEL_PATH, map_location=device, weights_only=True)
)
model.eval()

all_probs, all_targets = [], []

with torch.no_grad():
    for batch in val_loader:
        batch = batch.to(device)

        logits = model(batch)
        probs = torch.sigmoid(logits)

        all_probs.append(probs.cpu().numpy())
        all_targets.append(batch.y.cpu().numpy())

val_probs = np.concatenate(all_probs)
val_targets = np.concatenate(all_targets)

precision, recall, thresholds = precision_recall_curve(
    val_targets, val_probs
)

f1 = 2 * precision * recall / (precision + recall + 1e-8)

# Last precision/recall values do not have a matching threshold
best_index = np.argmax(f1[:-1])
best_threshold = thresholds[best_index]

print("Best epoch:", 7)
print("Saved PR-AUC:", 0.3840)
print("Best threshold:", round(float(best_threshold), 4))
print("Validation PR-AUC:",
      round(average_precision_score(val_targets, val_probs), 4))
print("Precision:", round(float(precision[best_index]), 4))
print("Recall:", round(float(recall[best_index]), 4))
print("F1:", round(float(f1[best_index]), 4))

Best epoch: 7
Saved PR-AUC: 0.384
Best threshold: 0.8609
Validation PR-AUC: 0.5661
Precision: 0.6875
Recall: 0.6416
F1: 0.6638


In [ ]:
best_threshold_v2 = 0.8609

In [ ]:
#3 evalutaing on test

test_probs, test_targets = [], []

model.eval()

with torch.no_grad():
    for batch in test_loader:
        batch = batch.to(device)
        probs = torch.sigmoid(model(batch))

        test_probs.append(probs.cpu().numpy())
        test_targets.append(batch.y.cpu().numpy())

test_probs = np.concatenate(test_probs)
test_targets = np.concatenate(test_targets).astype(int)

test_predictions = (
    test_probs >= best_threshold_v2
).astype(int)

tn, fp, fn, tp = confusion_matrix(
    test_targets, test_predictions
).ravel()

print("Test PR-AUC:",
      average_precision_score(test_targets, test_probs))
print("Test ROC-AUC:",
      roc_auc_score(test_targets, test_probs))
print("Precision:",
      precision_score(test_targets, test_predictions, zero_division=0))
print("Recall:",
      recall_score(test_targets, test_predictions, zero_division=0))
print("F1:",
      f1_score(test_targets, test_predictions, zero_division=0))
print("TP:", tp, "| FP:", fp, "| FN:", fn, "| TN:", tn)

Test PR-AUC: 0.7393193708544149
Test ROC-AUC: 0.9916013902649657
Precision: 0.5658073270013568
Recall: 0.8705636743215032
F1: 0.6858552631578947
TP: 834 | FP: 640 | FN: 124 | TN: 230490


In [ ]:
import json
import shutil

V2_MODEL_PATH = os.path.join(
    DATA_DIR, "models", "binary_graphsage_v2.pt"
)

shutil.copy2(BEST_MODEL_PATH, V2_MODEL_PATH)

v2_results = {
    "best_epoch": 1,
    "positive_weight": float(positive_weight),
    "validation_pr_auc": 0.5661,
    "validation_threshold": float(best_threshold_v2),
    "test_pr_auc": 0.7393193708544149,
    "test_roc_auc": 0.9916013902649657,
    "test_precision": 0.5658073270013568,
    "test_recall": 0.8705636743215032,
    "test_f1": 0.6858552631578947,
    "true_positives": 834,
    "false_positives": 640,
    "false_negatives": 124,
    "true_negatives": 230490
}

with open(
    os.path.join(DATA_DIR, "binary_graphsage_v2.json"), "w"
) as file:
    json.dump(v2_results, file, indent=2)

print("Model saved:", V2_MODEL_PATH)

Model saved: /content/drive/MyDrive/Threat-detection/gnn_edge_data/models/binary_graphsage_v2.pt
